In [1]:
# import

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import torch

from src.models.activations import sigmoid, relu, tanh, softmax
from src.models.attentions import ManualLinear, BahdanauAttention, LuongAttention
from src.models.layers import Embedding, VanillaRNN, LSTM, GRU
from src.models.encoder import Encoder
from src.models.decoder import Decoder
from src.models.seq2seq import Seq2Seq
from src.data.vocab import Vocabulary
from src.data.vi_tokenizer import tokenize_vi
from src.data.en_tokenizer import EnglishBPETokenizer
from src.data.dataset import TranslationDataset, CollateBatch, get_dataloader

In [2]:
# test activations

x = torch.tensor([-1.0, 0.0, 1.0])

assert torch.allclose(sigmoid(x), torch.sigmoid(x))
assert torch.allclose(relu(x), torch.relu(x))
assert torch.allclose(tanh(x), torch.tanh(x))

scores = torch.tensor([[1.0, 2.0, 3.0]])
probs = softmax(scores)

assert probs.shape == scores.shape
assert torch.allclose(probs.sum(dim=-1), torch.tensor([1.0]))

print("activations passed")

activations passed


In [3]:
# test ManualLinear

layer = ManualLinear(in_features=4, out_features=3)
x = torch.randn(2, 4)

y = layer(x)

assert y.shape == (2, 3)
assert layer.weight.shape == (3, 4)
assert layer.bias.shape == (3,)

print("ManualLinear passed")

ManualLinear passed


In [4]:
# test attention Bahdanau

batch_size = 2
src_len = 5
hidden_size = 8
attention_dim = 6

attention = BahdanauAttention(
    encoder_hidden_dim=hidden_size,
    decoder_hidden_dim=hidden_size,
    attention_dim=attention_dim
)

decoder_hidden = torch.randn(batch_size, hidden_size)
encoder_outputs = torch.randn(batch_size, src_len, hidden_size)

context, weights = attention(decoder_hidden, encoder_outputs)

assert context.shape == (batch_size, hidden_size)
assert weights.shape == (batch_size, src_len)
assert torch.allclose(weights.sum(dim=1), torch.ones(batch_size), atol=1e-6)

print("BahdanauAttention passed")

BahdanauAttention passed


In [5]:
# test attention mask

mask = torch.tensor([
    [False, False, True, True, True],
    [False, False, False, True, True]
])

context, weights = attention(decoder_hidden, encoder_outputs, mask=mask)

assert context.shape == (batch_size, hidden_size)
assert weights.shape == (batch_size, src_len)
assert torch.allclose(weights[mask], torch.zeros_like(weights[mask]), atol=1e-6)
assert torch.allclose(weights.sum(dim=1), torch.ones(batch_size), atol=1e-6)

print("attention mask passed")

attention mask passed


In [6]:
# test Luong attention

for method in ["dot", "general", "concat"]:
    attention = LuongAttention(
        encoder_hidden_dim=hidden_size,
        decoder_hidden_dim=hidden_size,
        score_method=method
    )

    context, weights = attention(decoder_hidden, encoder_outputs)

    assert context.shape == (batch_size, hidden_size)
    assert weights.shape == (batch_size, src_len)
    assert torch.allclose(weights.sum(dim=1), torch.ones(batch_size), atol=1e-6)

print("LuongAttention passed")

LuongAttention passed


In [7]:
# test Embedding

vocab_size = 20
embedding_dim = 7

embedding = Embedding(vocab_size, embedding_dim)

single = embedding(3)
many = embedding(torch.tensor([1, 2, 3, 4]))

assert single.shape == (embedding_dim,)
assert many.shape == (4, embedding_dim)

print("Embedding passed")

Embedding passed


In [8]:
# test VanillaRNN

seq_len = 4
input_size = 6
hidden_size = 8

rnn = VanillaRNN(input_size=input_size, hidden_size=hidden_size)
inputs = torch.randn(seq_len, input_size)

outputs, final_state = rnn(inputs)

assert len(outputs) == seq_len
assert outputs[0].shape == (hidden_size, 1)
assert final_state.shape == (hidden_size, 1)

print("VanillaRNN passed")

VanillaRNN passed


In [9]:
# test LSTM

seq_len = 4
input_size = 6
hidden_size = 8

lstm = LSTM(input_size=input_size, hidden_size=hidden_size)
inputs = torch.randn(seq_len, input_size)

outputs, final_state = lstm(inputs)
h_n, c_n = final_state

assert len(outputs) == seq_len
assert outputs[0].shape == (hidden_size, 1)
assert h_n.shape == (hidden_size, 1)
assert c_n.shape == (hidden_size, 1)

print("LSTM passed")

LSTM passed


In [10]:
# test GRU

seq_len = 4
input_size = 6
hidden_size = 8

gru = GRU(input_size=input_size, hidden_size=hidden_size)

h_t = torch.zeros(1, hidden_size)
outputs = []

for _ in range(seq_len):
    x_t = torch.randn(1, input_size)
    h_t = gru.step(x_t, h_t)
    outputs.append(h_t)

assert len(outputs) == seq_len
assert outputs[0].shape == (1, hidden_size)
assert h_t.shape == (1, hidden_size)

print("GRU step passed")

GRU step passed


In [11]:
# test data pipeline

src_texts = [
    "I am happy.",
    "You are a student.",
    "I like machine translation."
]

trg_texts = [
    "Tôi vui.",
    "Bạn là học sinh.",
    "Tôi thích dịch máy."
]

en_tokenizer = EnglishBPETokenizer(num_merges=20)
en_tokenizer.train(src_texts)

tokenizer_src = en_tokenizer.encode
tokenizer_trg = tokenize_vi

src_tokens = [["<sos>"] + tokenizer_src(text) + ["<eos>"] for text in src_texts]
trg_tokens = [["<sos>"] + tokenizer_trg(text) + ["<eos>"] for text in trg_texts]

src_vocab = Vocabulary(freq_threshold=1)
trg_vocab = Vocabulary(freq_threshold=1)

src_vocab.build_vocabulary(src_tokens)
trg_vocab.build_vocabulary(trg_tokens)

dataset = TranslationDataset(
    src_texts,
    trg_texts,
    src_vocab,
    trg_vocab,
    tokenizer_src,
    tokenizer_trg
)

assert len(dataset) == len(src_texts)

src_ids, trg_ids = dataset[0]

assert src_ids[0] == src_vocab.stoi["<sos>"]
assert src_ids[-1] == src_vocab.stoi["<eos>"]
assert trg_ids[0] == trg_vocab.stoi["<sos>"]
assert trg_ids[-1] == trg_vocab.stoi["<eos>"]

print("data pipeline passed")

data pipeline passed


In [12]:
# test collate batch

collate = CollateBatch(pad_idx=src_vocab.stoi["<pad>"])
batch = [dataset[i] for i in range(len(dataset))]

src_batch, trg_batch, src_mask, trg_mask = collate(batch)

assert src_batch.dim() == 2
assert trg_batch.dim() == 2
assert src_batch.shape == src_mask.shape
assert trg_batch.shape == trg_mask.shape
assert src_mask.dtype == torch.bool
assert trg_mask.dtype == torch.bool

print("collate batch passed")

collate batch passed


In [13]:
# test dataloader

loader = get_dataloader(
    src_texts,
    trg_texts,
    src_vocab,
    trg_vocab,
    tokenizer_src,
    tokenizer_trg,
    batch_size=2
)

src_batch, trg_batch, src_mask, trg_mask = next(iter(loader))

assert src_batch.shape[0] <= 2
assert trg_batch.shape[0] <= 2
assert src_mask.shape == src_batch.shape
assert trg_mask.shape == trg_batch.shape

print("dataloader passed")

dataloader passed


In [14]:
# test encoder GRU

batch_size = 3
src_len = 5
src_vocab_size = src_vocab_size = len(src_vocab.stoi)
embed_dim = 8
hidden_size = 16
num_layers = 1

encoder = Encoder(
    vocab_size=src_vocab_size,
    embed_dim=embed_dim,
    hidden_size=hidden_size,
    num_layers=num_layers,
    cell_type="gru",
    dropout=0.0
)

src = torch.randint(0, src_vocab_size, (batch_size, src_len))

encoder_outputs, final_hidden = encoder(src)

assert encoder_outputs.shape == (batch_size, src_len, hidden_size)
assert final_hidden.shape == (num_layers, batch_size, hidden_size)

print("encoder GRU passed")

encoder GRU passed


In [15]:
# test decoder GRU without attention

tgt_vocab_size = len(trg_vocab)
tgt_len = 6

decoder = Decoder(
    vocab_size=tgt_vocab_size,
    embed_dim=8,
    hidden_size=hidden_size,
    num_layers=1,
    cell_type="gru",
    attention=None,
    dropout=0.0,
    eos_token_id=trg_vocab.stoi["<eos>"]
)

tgt = torch.randint(0, tgt_vocab_size, (batch_size, tgt_len))
tgt[:, 0] = trg_vocab.stoi["<sos>"]

logits, attns = decoder(
    tgt=tgt,
    final_hidden=final_hidden,
    encoder_outputs=encoder_outputs,
    teacher_forcing_ratio=1.0
)

assert logits.shape == (batch_size, tgt_len - 1, tgt_vocab_size)
assert attns is None

print("decoder GRU without attention passed")

decoder GRU without attention passed


In [16]:
# test decoder GRU with Bahdanau attention

attention = BahdanauAttention(
    encoder_hidden_dim=hidden_size,
    decoder_hidden_dim=hidden_size,
    attention_dim=12
)

decoder = Decoder(
    vocab_size=tgt_vocab_size,
    embed_dim=8,
    hidden_size=hidden_size,
    num_layers=1,
    cell_type="gru",
    attention=attention,
    dropout=0.0,
    eos_token_id=trg_vocab.stoi["<eos>"]
)

src_mask = src.eq(src_vocab.stoi["<pad>"])

logits, attns = decoder(
    tgt=tgt,
    final_hidden=final_hidden,
    encoder_outputs=encoder_outputs,
    src_mask=src_mask,
    teacher_forcing_ratio=1.0
)

assert logits.shape == (batch_size, tgt_len - 1, tgt_vocab_size)
assert attns.shape == (batch_size, tgt_len - 1, src_len)
assert torch.allclose(attns.sum(dim=-1), torch.ones(batch_size, tgt_len - 1), atol=1e-6)

print("decoder GRU with attention passed")

decoder GRU with attention passed


In [17]:
# test Seq2Seq GRU

encoder = Encoder(
    vocab_size=src_vocab_size,
    embed_dim=8,
    hidden_size=hidden_size,
    num_layers=1,
    cell_type="gru",
    dropout=0.0
)

attention = BahdanauAttention(
    encoder_hidden_dim=hidden_size,
    decoder_hidden_dim=hidden_size,
    attention_dim=12
)

decoder = Decoder(
    vocab_size=tgt_vocab_size,
    embed_dim=8,
    hidden_size=hidden_size,
    num_layers=1,
    cell_type="gru",
    attention=attention,
    dropout=0.0,
    eos_token_id=trg_vocab.stoi["<eos>"]
)

model = Seq2Seq(
    encoder=encoder,
    decoder=decoder,
    src_pad_idx=src_vocab.stoi["<pad>"]
)

logits, attns = model(
    src=src,
    tgt=tgt,
    teacher_forcing_ratio=1.0
)

assert logits.shape == (batch_size, tgt_len - 1, tgt_vocab_size)
assert attns.shape == (batch_size, tgt_len - 1, src_len)

print("Seq2Seq GRU passed")

Seq2Seq GRU passed


In [18]:
# test greedy decode

logits, attns = model.greedy_decode(
    src=src,
    bos_token_id=trg_vocab.stoi["<sos>"],
    max_length=4
)

assert logits.shape[0] == batch_size
assert logits.shape[2] == tgt_vocab_size
assert logits.shape[1] <= 4

if attns is not None:
    assert attns.shape[0] == batch_size
    assert attns.shape[2] == src_len

print("greedy decode passed")

greedy decode passed


In [19]:
# test invalid options

try:
    Encoder(
        vocab_size=10,
        embed_dim=4,
        hidden_size=8,
        cell_type="invalid"
    )
    raise AssertionError("Encoder should raise ValueError")
except ValueError:
    pass

try:
    LuongAttention(
        encoder_hidden_dim=8,
        decoder_hidden_dim=8,
        score_method="invalid"
    )
    raise AssertionError("LuongAttention should raise ValueError")
except ValueError:
    pass

print("invalid options passed")

invalid options passed
